# Shared code for the SVR notebooks

Registry, feature sets, and scoring helpers shared by `3.2a PredictionSVM_Standard.ipynb` (direct
target) and `3.2b PredictionSVM_Residual.ipynb` (residual-on-baseline target). Extracted here because
a cell-by-cell diff between the two notebooks showed these pieces were byte-identical or functionally
identical (only docstrings differed) — kept in two places, a fix like the residual notebook's
`SVR_MAX_ITER` cap only landing in one of them was exactly the kind of drift this notebook removes.

The two notebooks' actual training logic (`GridSearchCV` in `3.2a` vs. a manual K-fold loop in `3.2b`,
forced by `3.2b`'s need to reconstruct counts from a residual prediction using `base_demand`) is
**not** here — it is fundamentally different between the two and stays in each notebook as explicit,
separate code.

Both notebooks pull this in with IPython's `%run "3.2 PredictionSVM_CommonGround.ipynb"` — it
executes every code cell below directly into the caller's namespace (the same effect as
`from module import *`), so no separate `.py` file is needed and this stays a notebook like everything
else in the pipeline. It is never registered in `report.qmd` and has no `#| label:` cells: it is not
itself a report figure/table, only a dependency of the two that are.

In [ ]:
import itertools
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

TARGET = "Total_Trip_Start"
RANDOM_STATE = 42
SPLIT_DIR = Path("..") / "data" / "prediction_split"

# Training budget. SVR is O(n^2) to fit and O(n_SV * n) to predict, so the raw splits (up to 6.2M
# training rows) are far out of reach. We fit on a small balanced sample and select on a validation
# subsample; only the winning model per level touches the full test split. Identical across both
# notebooks so their results stay directly comparable.
N_SCAN = 4_000       # per class, for the kernel scan  -> 8k rows
N_GRID = 8_000       # per class, for the grid search  -> 16k rows
N_VAL_SUBSAMPLE = 50_000

KERNELS = ["linear", "poly", "rbf"]


## The eight levels

Four spatial levels (H3 res-7, H3 res-8, census tract, community area) crossed with two temporal
levels (hourly, daily). Each split lives at
`data/prediction_split/<spatial>/<temporal>/{train,val,test}.parquet`; the spatial level determines
the name of the key column.

In [ ]:
SPATIAL_LEVELS = {
    "h3_7":      "h3_index_7",
    "h3_8":      "h3_index_8",
    "census":    "geoid10",
    "community": "commarea",
}
TEMPORAL_LEVELS = ["hourly", "daily"]

LEVELS = list(itertools.product(SPATIAL_LEVELS, TEMPORAL_LEVELS))


def load_splits(spatial, temporal):
    """Load train/val/test for one (spatial, temporal) level."""
    base = SPLIT_DIR / spatial / temporal
    return tuple(pd.read_parquet(base / f"{s}.parquet") for s in ("train", "val", "test"))


## Feature sets

Four candidate feature sets per level: a `basic` calendar + location + history block, then POI
counts and weather layered on top. Daily grids have no hour features — a daily row is floored to
midnight, so `hour_sin`/`hour_cos` would be constants — so the feature set is derived from the
temporal level rather than being a single global list.

In [ ]:
HOUR = ["hour_sin", "hour_cos"]                    # hourly grids only
CALENDAR = ["month_sin", "month_cos", "is_weekend",
            "is_holiday", "is_near_holiday", "day_of_week"]
LOCATION = ["lat", "lon", "distance_to_loop"]
HISTORY = ["base_demand"]                          # train-only historical mean, computed in 3.1

POI_COLS = ["poi_cat_automotive", "poi_cat_civic_community", "poi_cat_education",
            "poi_cat_entertainment", "poi_cat_finance", "poi_cat_food_drink", "poi_cat_grocery",
            "poi_cat_health", "poi_cat_leisure_sports", "poi_cat_lodging", "poi_cat_nightlife",
            "poi_cat_services", "poi_cat_shopping", "poi_cat_transport"]
WEATHER = ["2m_temp_c", "total_precip_mm", "snow_cov", "snow_depth", "wind_speed"]


def feature_sets(temporal):
    """The four candidate feature sets for a given temporal level."""
    basic = (HOUR if temporal == "hourly" else []) + CALENDAR + LOCATION + HISTORY
    return {
        "basic": basic,
        "basic+poi": basic + POI_COLS,
        "basic+weather": basic + WEATHER,
        "basic+poi+weather": basic + POI_COLS + WEATHER,
    }


## Scoring helpers

- **log1p target.** Demand is counts with a long right tail. We fit on `log1p(y)` and invert with
  `expm1`, clipped at zero.
- **Balanced sample.** At res-8 hourly the target is 96.9% zeros; a uniform sample would be almost
  all zeros and the SVR would learn to predict nothing. We take all non-zero rows up to a cap plus an
  equal number of zeros. Where a level is *not* zero-inflated (`community/daily` is only 0.6% zeros)
  the zero class simply runs out and the sample is non-zero-heavy — that is the correct behaviour.
- **Metrics in count space**, not log space, so MAE is "trips per cell-period".
- **Skill score** is scale-free so it is comparable across levels — raw MAE is not, since mean demand
  ranges from 0.73 per cell-hour at h3_8/hourly to 213 per area-day at community/daily, a 300x gap.

In [ ]:
LOG_CEILING = 20.0   # expm1(20) ~ 4.9e8 pickups in one cell-period: already absurd


def to_counts(pred_log):
    """Invert log1p and clip negative demand to 0.

    The log prediction is clipped first. An unbounded linear kernel can predict a log-demand of
    several hundred, and expm1 of that is +inf -- which then propagates into r2_score as a crash
    rather than a bad score. Clipping keeps the linear kernel's failure *finite and reportable*
    without touching any prediction a sane model would make.
    """
    return np.clip(np.expm1(np.clip(pred_log, -LOG_CEILING, LOG_CEILING)), 0, None)


def score(y_true, pred_counts, name):
    return {
        "Model": name,
        "MAE": mean_absolute_error(y_true, pred_counts),
        "RMSE": np.sqrt(mean_squared_error(y_true, pred_counts)),
        "R2": r2_score(y_true, pred_counts),
    }


def balanced_sample(df, n_per_class, seed=RANDOM_STATE):
    """All non-zero rows (capped) plus an equal number of zero rows, shuffled.

    Where a level is not zero-inflated the zero class runs out and min() caps it -- the sample is then
    simply non-zero-heavy, which is what we want.
    """
    nz = df[df[TARGET] > 0]
    z = df[df[TARGET] == 0]
    return pd.concat([
        nz.sample(n=min(n_per_class, len(nz)), random_state=seed),
        z.sample(n=min(n_per_class, len(z)), random_state=seed),
    ]).sample(frac=1, random_state=seed)


def skill(model_metrics, baseline_metrics):
    """Fraction of the baseline's error the model removes. 0 = no better than baseline, 1 = perfect.

    Raw MAE cannot be compared across levels -- mean demand is 0.73 per cell-hour at h3_8/hourly but
    213 per area-day at community/daily, a 300x scale gap, so the finest grid would always 'win' on
    MAE simply by having less to be wrong about. Skill is scale-free: it asks whether the SVR beats
    the historical-mean baseline ON ITS OWN GROUND, which is comparable between levels.
    """
    return {
        "skill_MAE": 1 - model_metrics["MAE"] / baseline_metrics["MAE"],
        "skill_RMSE": 1 - model_metrics["RMSE"] / baseline_metrics["RMSE"],
    }
